In [2]:
#imports
import pandas as pd
import numpy as np
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

## Read in adult census data and perform data preprocessing

In [3]:
# importing the data
adult = pd.read_csv('/Users/donyabehroozi/Documents/gsb545/GSB-545/In Class Assignments/adult.csv')
adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
#data preprocessing

# replace ? with np.nan
adult = adult.replace("?", np.nan)
adult.head()

# convert target variable to binary
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

# convert gender to 0/1 (doesn't need categorical encoding since it's binary)
if "gender" in adult.columns:
    adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)

# drop the fnlwgt variable as it is not useful for modeling
adult.drop(columns=["fnlwgt"], inplace=True)  
    
adult.head(20)

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0
5,34,Private,10th,6,Never-married,Other-service,Not-in-family,White,1,0,0,30,United-States,0
6,29,NaN,HS-grad,9,Never-married,NaN,Unmarried,Black,1,0,0,40,United-States,0
7,63,Self-emp-not-inc,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,1,3103,0,32,United-States,1
8,24,Private,Some-college,10,Never-married,Other-service,Unmarried,White,0,0,0,40,United-States,0
9,55,Private,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,1,0,0,10,United-States,0


In [5]:
#feature engineering

#net capital activity
adult['capital_net'] = adult['capital-gain'] - adult['capital-loss'] 

#age and education interaction
adult['age_education_interaction'] = adult['age'] * adult['educational-num']

#log transformation of net capital activity to help models handle extreme values
capital_net_clipped = adult['capital_net'].clip(lower=0)
adult['capital_net_log'] = np.log1p(capital_net_clipped)

#age and marriage interaction
adult['age_married'] = adult['age'] * (adult['marital-status'] == 'Married-civ-spouse').astype(int)

adult.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,capital_net,age_education_interaction,capital_net_log,age_married
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,0,175,0.000000,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,0,342,0.000000,38
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1,0,336,0.000000,28
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1,7688,440,8.947546,44
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0,0,180,0.000000,0


In [17]:
#count NAs
adult.isna().sum()

age                             0
workclass                    2799
education                       0
educational-num                 0
marital-status                  0
occupation                   2809
relationship                    0
race                            0
gender                          0
capital-gain                    0
capital-loss                    0
hours-per-week                  0
native-country                857
income                          0
capital_net                     0
age_education_interaction       0
capital_net_log                 0
age_married                     0
dtype: int64

In [6]:
#impute missing values with "unknown"
for col in ['workclass', 'occupation', 'native-country']:
    adult[col] = adult[col].fillna('Unknown')

adult.head(20)

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,capital_net,age_education_interaction,capital_net_log,age_married
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,0,175,0.000000,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,0,342,0.000000,38
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1,0,336,0.000000,28
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1,7688,440,8.947546,44
4,18,Unknown,Some-college,10,Never-married,Unknown,Own-child,White,0,0,0,30,United-States,0,0,180,0.000000,0
5,34,Private,10th,6,Never-married,Other-service,Not-in-family,White,1,0,0,30,United-States,0,0,204,0.000000,0
6,29,Unknown,HS-grad,9,Never-married,Unknown,Unmarried,Black,1,0,0,40,United-States,0,0,261,0.000000,0
7,63,Self-emp-not-inc,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,1,3103,0,32,United-States,1,3103,945,8.040447,63
8,24,Private,Some-college,10,Never-married,Other-service,Unmarried,White,0,0,0,40,United-States,0,0,240,0.000000,0
9,55,Private,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,1,0,0,10,United-States,0,0,220,0.000000,55


In [7]:
# defining X
num_cols = ['age', 'educational-num', 'capital-gain', 'capital-loss',
            'hours-per-week', 'capital_net', 'capital_net_log', 
            'age_education_interaction', 'age_married']

cat_cols = ['workclass', 'education', 'marital-status',
            'occupation', 'relationship', 'race', 'native-country']

X = pd.concat([
    adult[num_cols],
    pd.get_dummies(adult[cat_cols], drop_first=True).astype(int)
], axis=1)

# defining y
y = adult['income'].to_numpy() 

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=321)

# scale numeric columns only
min_max_scaler = MinMaxScaler()
X_train[num_cols] = min_max_scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = min_max_scaler.transform(X_test[num_cols])

print(X_train.head())
print(y_train[:5])

            age  educational-num  capital-gain  capital-loss  hours-per-week  \
48353  0.027397         0.533333           0.0      0.000000        0.397959   
8310   0.301370         0.666667           0.0      0.000000        0.500000   
6063   0.287671         0.733333           0.0      0.000000        0.377551   
8229   0.068493         0.333333           0.0      0.000000        0.397959   
20762  0.109589         0.800000           0.0      0.399679        0.397959   

       capital_net  capital_net_log  age_education_interaction  age_married  \
48353     0.041742              0.0                   0.114200          0.0   
8310      0.041742              0.0                   0.308039          0.0   
6063      0.041742              0.0                   0.328325          0.0   
8229      0.041742              0.0                   0.084899          0.0   
20762     0.025059              0.0                   0.229902          0.0   

       workclass_Local-gov  ...  native-coun

## Baseline neural network (Funnel architecture, 2 hidden layers, 64 -> 32 -> 1)

In [8]:
#set random seed
tf.random.set_seed(321)

In [21]:
#construct the model
inputs = keras.Input(shape=(102,)) #102 inputs, input layer
x = layers.Dense(64, activation='relu')(inputs) #hidden layer 1
x = layers.Dense(32, activation='relu')(x) #hidden layer 2
outputs = layers.Dense(1, activation='sigmoid')(x) #output layer (sigmoid for binary outcome)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_model")

In [22]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 539us/step - auc: 0.8803 - loss: 0.3550 - val_auc: 0.8952 - val_loss: 0.3349
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - auc: 0.9021 - loss: 0.3249 - val_auc: 0.8993 - val_loss: 0.3282
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 413us/step - auc: 0.9067 - loss: 0.3177 - val_auc: 0.9020 - val_loss: 0.3242
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 479us/step - auc: 0.9103 - loss: 0.3118 - val_auc: 0.9037 - val_loss: 0.3217
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - auc: 0.9133 - loss: 0.3069 - val_auc: 0.9049 - val_loss: 0.3200
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 361us/step - auc: 0.9158 - loss: 0.3027 - val_auc: 0.9059 - val_loss: 0.3190
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 361us/step - auc: 0.9179 - loss: 0.2992 - val_auc: 0.9066 - val_loss: 0.3187
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 360us/step - auc: 0.9196 - loss: 0.2962 - val_auc: 0.9064 - val_loss: 0.3197
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [23]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 205us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.9007


# Experiment 1: Architecture change

## Flat architecture (102 -> 64 -> 64 -> 1)

In [24]:
inputs = keras.Input(shape=(102,))
x = layers.Dense(64, activation='relu')(inputs)  # hidden layer 1
x = layers.Dense(64, activation='relu')(x)        # hidden layer 2
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_flat")

In [25]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 454us/step - auc: 0.8834 - loss: 0.3510 - val_auc: 0.8961 - val_loss: 0.3337
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 382us/step - auc: 0.9027 - loss: 0.3243 - val_auc: 0.8996 - val_loss: 0.3282
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 371us/step - auc: 0.9069 - loss: 0.3177 - val_auc: 0.9023 - val_loss: 0.3244
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 357us/step - auc: 0.9100 - loss: 0.3124 - val_auc: 0.9040 - val_loss: 0.3216
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 354us/step - auc: 0.9128 - loss: 0.3078 - val_auc: 0.9049 - val_loss: 0.3206
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 385us/step - auc: 0.9152 - loss: 0.3038 - val_auc: 0.9061 - val_loss: 0.3196
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 373us/step - auc: 0.9173 - loss: 0.3004 - val_auc: 0.9067 - val_loss: 0.3186
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 358us/step - auc: 0.9192 - loss: 0.2971 - val_auc: 0.9066 - val_loss: 0.3191
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [26]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 208us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.9006


## Expanding architecture (102 -> 32 -> 64 -> 1)

In [27]:
inputs = keras.Input(shape=(102,))
x = layers.Dense(32, activation='relu')(inputs)  
x = layers.Dense(64, activation='relu')(x)       
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_expanding")

In [28]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 473us/step - auc: 0.8798 - loss: 0.3554 - val_auc: 0.8950 - val_loss: 0.3353
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 419us/step - auc: 0.9016 - loss: 0.3261 - val_auc: 0.8992 - val_loss: 0.3292
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - auc: 0.9059 - loss: 0.3193 - val_auc: 0.9021 - val_loss: 0.3247
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 358us/step - auc: 0.9092 - loss: 0.3140 - val_auc: 0.9041 - val_loss: 0.3216
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 352us/step - auc: 0.9117 - loss: 0.3098 - val_auc: 0.9053 - val_loss: 0.3192
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - auc: 0.9137 - loss: 0.3065 - val_auc: 0.9064 - val_loss: 0.3177
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 355us/step - auc: 0.9154 - loss: 0.3035 - val_auc: 0.9065 - val_loss: 0.3172
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 351us/step - auc: 0.9169 - loss: 0.3011 - val_auc: 0.9068 - val_loss: 0.3169
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [29]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 205us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.9054


# Experiment 2: Different number of layers

## Single hidden layer

In [ ]:
#construct the model
inputs = keras.Input(shape=(102,)) #102 inputs, input layer
x = layers.Dense(64, activation='relu')(inputs) #hidden layer 1
outputs = layers.Dense(1, activation='sigmoid')(x) #output layer (sigmoid for binary outcome)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_model_sl")

In [31]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 445us/step - auc: 0.8763 - loss: 0.3601 - val_auc: 0.8936 - val_loss: 0.3374
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - auc: 0.9006 - loss: 0.3273 - val_auc: 0.8971 - val_loss: 0.3320
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 337us/step - auc: 0.9039 - loss: 0.3222 - val_auc: 0.8994 - val_loss: 0.3283
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 337us/step - auc: 0.9063 - loss: 0.3183 - val_auc: 0.9013 - val_loss: 0.3255
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 357us/step - auc: 0.9084 - loss: 0.3149 - val_auc: 0.9027 - val_loss: 0.3234
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 396us/step - auc: 0.9102 - loss: 0.3121 - val_auc: 0.9037 - val_loss: 0.3218
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 336us/step - auc: 0.9119 - loss: 0.3094 - val_auc: 0.9046 - val_loss: 0.3203
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - auc: 0.9133 - loss: 0.3071 - val_auc: 0.9056 - val_loss: 0.3189
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [32]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 184us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.9077


## 3 hidden layers, Funnel

In [ ]:
#construct the model
inputs = keras.Input(shape=(102,)) #102 inputs, input layer
x = layers.Dense(128, activation='relu')(inputs) #hidden layer 1
x = layers.Dense(64, activation='relu')(x) #hidden layer 2
x = layers.Dense(32, activation='relu')(x) #hidden layer 3
outputs = layers.Dense(1, activation='sigmoid')(x) #output layer (sigmoid for binary outcome)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_model_funnel3")

In [34]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 512us/step - auc: 0.8865 - loss: 0.3471 - val_auc: 0.8967 - val_loss: 0.3328
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 507us/step - auc: 0.9039 - loss: 0.3222 - val_auc: 0.9008 - val_loss: 0.3261
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 423us/step - auc: 0.9093 - loss: 0.3134 - val_auc: 0.9032 - val_loss: 0.3225
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 419us/step - auc: 0.9136 - loss: 0.3064 - val_auc: 0.9043 - val_loss: 0.3214
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 420us/step - auc: 0.9171 - loss: 0.3005 - val_auc: 0.9049 - val_loss: 0.3211
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 430us/step - auc: 0.9204 - loss: 0.2950 - val_auc: 0.9047 - val_loss: 0.3226
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 418us/step - auc: 0.9230 - loss: 0.2904 - val_auc: 0.9042 - val_loss: 0.3249
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 420us/step - auc: 0.9254 - loss: 0.2860 - val_auc: 0.9032 - val_loss: 0.3291
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [35]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 221us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.8841


## 3 hidden layers, Flat

In [48]:
#construct the model
inputs = keras.Input(shape=(102,)) #102 inputs, input layer
x = layers.Dense(64, activation='relu')(inputs) #hidden layer 1
x = layers.Dense(64, activation='relu')(x) #hidden layer 2
x = layers.Dense(64, activation='relu')(x) #hidden layer 3
outputs = layers.Dense(1, activation='sigmoid')(x) #output layer (sigmoid for binary outcome)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_model_flat3")

In [49]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 513us/step - auc: 0.8871 - loss: 0.3463 - val_auc: 0.8964 - val_loss: 0.3335
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - auc: 0.9033 - loss: 0.3231 - val_auc: 0.8999 - val_loss: 0.3277
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - auc: 0.9077 - loss: 0.3161 - val_auc: 0.9024 - val_loss: 0.3243
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 399us/step - auc: 0.9114 - loss: 0.3100 - val_auc: 0.9034 - val_loss: 0.3233
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 387us/step - auc: 0.9148 - loss: 0.3045 - val_auc: 0.9037 - val_loss: 0.3237
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 389us/step - auc: 0.9175 - loss: 0.3000 - val_auc: 0.9041 - val_loss: 0.3248
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 391us/step - auc: 0.9199 - loss: 0.2956 - val_auc: 0.9040 - val_loss: 0.3261
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 389us/step - auc: 0.9221 - loss: 0.2915 - val_auc: 0.9040 - val_loss: 0.3270
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [50]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 221us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.8918


## 3 hidden layers, Expanding

In [45]:
#construct the model
inputs = keras.Input(shape=(102,)) #102 inputs, input layer
x = layers.Dense(32, activation='relu')(inputs) #hidden layer 1
x = layers.Dense(64, activation='relu')(x) #hidden layer 2
x = layers.Dense(128, activation='relu')(x) #hidden layer 3
outputs = layers.Dense(1, activation='sigmoid')(x) #output layer (sigmoid for binary outcome)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_model_exp3")

In [46]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 486us/step - auc: 0.8807 - loss: 0.3544 - val_auc: 0.8947 - val_loss: 0.3361
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 397us/step - auc: 0.9009 - loss: 0.3268 - val_auc: 0.8980 - val_loss: 0.3310
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 391us/step - auc: 0.9049 - loss: 0.3203 - val_auc: 0.9005 - val_loss: 0.3272
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 390us/step - auc: 0.9082 - loss: 0.3151 - val_auc: 0.9019 - val_loss: 0.3255
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 391us/step - auc: 0.9114 - loss: 0.3101 - val_auc: 0.9035 - val_loss: 0.3238
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 393us/step - auc: 0.9141 - loss: 0.3056 - val_auc: 0.9040 - val_loss: 0.3237
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 393us/step - auc: 0.9161 - loss: 0.3022 - val_auc: 0.9043 - val_loss: 0.3239
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 392us/step - auc: 0.9179 - loss: 0.2990 - val_auc: 0.9040 - val_loss: 0.3248
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [47]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 230us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.8936


# Experiment 3: Differing dropout rates with baseline model

## No dropouts = baseline model, see results above

## Light dropout rate (0.2)

In [9]:
#construct the model
inputs = keras.Input(shape=(102,))
x = layers.Dense(64, activation='relu')(inputs)  # hidden layer 1
x = layers.Dropout(0.2)(x)                        # drop 20% of neurons
x = layers.Dense(32, activation='relu')(x)        # hidden layer 2
x = layers.Dropout(0.2)(x)                        # drop 20% of neurons
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_dropout_02")

In [10]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 465us/step - auc: 0.8694 - loss: 0.3694 - val_auc: 0.8936 - val_loss: 0.3376
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 403us/step - auc: 0.8964 - loss: 0.3339 - val_auc: 0.8972 - val_loss: 0.3333
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 371us/step - auc: 0.9014 - loss: 0.3262 - val_auc: 0.9000 - val_loss: 0.3282
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - auc: 0.9051 - loss: 0.3205 - val_auc: 0.9026 - val_loss: 0.3244
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - auc: 0.9076 - loss: 0.3162 - val_auc: 0.9039 - val_loss: 0.3229
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 461us/step - auc: 0.9098 - loss: 0.3128 - val_auc: 0.9053 - val_loss: 0.3205
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 467us/step - auc: 0.9114 - loss: 0.3104 - val_auc: 0.9059 - val_loss: 0.3192
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 372us/step - auc: 0.9129 - loss: 0.3078 - val_auc: 0.9061 - val_loss: 0.3195
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [11]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 208us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.9040


## Heavy dropout rate (0.5)

In [12]:
inputs = keras.Input(shape=(102,))
x = layers.Dense(64, activation='relu')(inputs)  # hidden layer 1
x = layers.Dropout(0.5)(x)                        # drop 50% of neurons
x = layers.Dense(32, activation='relu')(x)        # hidden layer 2
x = layers.Dropout(0.5)(x)                        # drop 50% of neurons
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="adult_income_dropout_05")

In [13]:
#defines loss, optimizer, metric, and fits model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

scores = model.evaluate(X_test, y_test, verbose = 1)

Epoch 1/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 1s 494us/step - auc: 0.8467 - loss: 0.3971 - val_auc: 0.8912 - val_loss: 0.3421
Epoch 2/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 375us/step - auc: 0.8867 - loss: 0.3504 - val_auc: 0.8956 - val_loss: 0.3356
Epoch 3/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 373us/step - auc: 0.8927 - loss: 0.3420 - val_auc: 0.8981 - val_loss: 0.3322
Epoch 4/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 372us/step - auc: 0.8948 - loss: 0.3384 - val_auc: 0.8998 - val_loss: 0.3291
Epoch 5/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 375us/step - auc: 0.8983 - loss: 0.3330 - val_auc: 0.9015 - val_loss: 0.3266
Epoch 6/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 374us/step - auc: 0.8999 - loss: 0.3296 - val_auc: 0.9015 - val_loss: 0.3258
Epoch 7/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 377us/step - auc: 0.9012 - loss: 0.3279 - val_auc: 0.9030 - val_loss: 0.3242
Epoch 8/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 0s 394us/step - auc: 0.9040 - loss: 0.3242 - val_auc: 0.9033 - val_loss: 0.3231
Epoch 9/20
977/977 ━━━━━━━━━━━━━━━━━━━━ 

In [14]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Test Set Evaluation:") 
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 201us/step

 Test Set Evaluation:
Test ROC AUC Score: 0.9069


# Model tuning with Optuna

In [15]:
def objective(trial):
    num_layers = trial.suggest_int("num_layers", 1, 3)  # number of hidden layers
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)  # log scale for LR
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])  # common batch sizes
    inputs = keras.Input(shape=(102,))  # 102 input features
    x = inputs

    for i in range(num_layers):
        units = trial.suggest_int(f"num_units_layer_{i+1}", 4, 128)  # neurons per layer
        x = layers.Dense(units, activation="relu")(x)  # hidden layer

    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)  # build model

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",  # watch validation loss
        patience=5,  # stop after 5 epochs without improvement
        restore_best_weights=True  # keep best model weights
    )

    history = model.fit(
        X_train,
        y_train,
        batch_size=batch_size,
        epochs=75,  # max training epochs
        validation_split=0.2,  # validation portion of training data
        verbose=0,  # suppress output during tuning
        callbacks=[early_stop]  # apply early stopping
    )

    return min(history.history["val_loss"])  # objective = best validation loss


study = optuna.create_study(direction="minimize")  # minimize validation loss
study.optimize(objective, n_trials=20)  # run 20 trials (20 models)

print("Best validation loss:", study.best_value)
print("Best parameters:", study.best_params)


[I 2026-05-19 12:05:26,641] A new study created in memory with name: no-name-3f8b1d41-0450-4d9b-a27a-2a1e564d21db
[I 2026-05-19 12:05:35,807] Trial 0 finished with value: 0.3152136504650116 and parameters: {'num_layers': 1, 'learning_rate': 0.0005464046577323495, 'batch_size': 64, 'num_units_layer_1': 69}. Best is trial 0 with value: 0.3152136504650116.
[I 2026-05-19 12:05:40,501] Trial 1 finished with value: 0.31629255414009094 and parameters: {'num_layers': 3, 'learning_rate': 0.0005395008012213594, 'batch_size': 64, 'num_units_layer_1': 30, 'num_units_layer_2': 83, 'num_units_layer_3': 5}. Best is trial 0 with value: 0.3152136504650116.
[I 2026-05-19 12:05:53,865] Trial 2 finished with value: 0.31602489948272705 and parameters: {'num_layers': 2, 'learning_rate': 0.00020423175713567328, 'batch_size': 32, 'num_units_layer_1': 29, 'num_units_layer_2': 100}. Best is trial 0 with value: 0.3152136504650116.
[I 2026-05-19 12:06:10,675] Trial 3 finished with value: 0.3178706467151642 and pa

Best validation loss: 0.31156104803085327
Best parameters: {'num_layers': 1, 'learning_rate': 0.002084513427543767, 'batch_size': 64, 'num_units_layer_1': 4}


In [16]:
# Building the best model from Optuna results
best_params = study.best_params
num_layers = best_params["num_layers"]
learning_rate = best_params["learning_rate"]
batch_size = best_params["batch_size"]  
inputs = keras.Input(shape=(102,))
x = inputs
for i in range(num_layers):
    units = best_params[f"num_units_layer_{i+1}"]
    x = layers.Dense(units, activation="relu")(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
best_model = keras.Model(inputs=inputs, outputs=outputs)
best_model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    metrics=[keras.metrics.AUC(name='auc')],
)   

history = best_model.fit(X_train, y_train, batch_size=batch_size, epochs=75, validation_split=0.2, verbose=1, callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])
scores = best_model.evaluate(X_test, y_test, verbose=1)  

Epoch 1/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 562us/step - auc: 0.8215 - loss: 0.4218 - val_auc: 0.8874 - val_loss: 0.3492
Epoch 2/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 350us/step - auc: 0.8960 - loss: 0.3352 - val_auc: 0.8928 - val_loss: 0.3396
Epoch 3/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 348us/step - auc: 0.8991 - loss: 0.3298 - val_auc: 0.8948 - val_loss: 0.3360
Epoch 4/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 354us/step - auc: 0.9010 - loss: 0.3268 - val_auc: 0.8963 - val_loss: 0.3336
Epoch 5/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 350us/step - auc: 0.9023 - loss: 0.3246 - val_auc: 0.8976 - val_loss: 0.3313
Epoch 6/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 353us/step - auc: 0.9034 - loss: 0.3229 - val_auc: 0.8988 - val_loss: 0.3296
Epoch 7/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 377us/step - auc: 0.9043 - loss: 0.3215 - val_auc: 0.8997 - val_loss: 0.3282
Epoch 8/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 0s 374us/step - auc: 0.9050 - loss: 0.3204 - val_auc: 0.9005 - val_loss: 0.3269
Epoch 9/75
489/489 ━━━━━━━━━━━━━━━━━━━━ 

In [18]:
# evaluate the best model using the test set
y_pred_prob = best_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\n Best Model Test Set Evaluation:")
print(f"Test ROC AUC Score: {roc_auc:.4f}")

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 164us/step

 Best Model Test Set Evaluation:
Test ROC AUC Score: 0.9051
